# Chapter 6 &mdash; Worked Example: Union, Minimization, Two Predicates

**Concept 11 of the Chapter 6 decomposition:** *Worked Example: Union, Minimization, and the Two Comparison Predicates*

Build two DFA, union them, minimize, and watch `iso_dfa` say False while `langeq_dfa` says True.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter6/Concept-Worked-Union-Minimization/Concept-Worked-Union-Minimization.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.AnimateDFA     import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


The full pipeline on one example:

1. build two DFA;
2. **union** them &mdash; the product construction inflates the state count;
3. **prune** the unreachable pairs;
4. **minimize** &mdash; the count collapses;
5. compare with **both** predicates.

The punchline is step 5: the raw union and the minimized union are
**language-equivalent** but **not isomorphic**. Confusing the two predicates is the
most common mistake in this chapter.

## 2. Definitions

### Two machines to combine

In [ ]:
even0 = md2mc('''DFA
IF : 0 -> Od
IF : 1 -> IF
Od : 0 -> IF
Od : 1 -> Od
''')
end1 = md2mc('''DFA
I : 0 -> I
I : 1 -> F
F : 0 -> I
F : 1 -> F
''')

### The pipeline, as one function

In [ ]:
def pipeline(A, B):
    U  = union_dfa(A, B)
    Up = pruneUnreach(U)
    Um = min_dfa(Up)
    return U, Up, Um

## 3. Tests

Sizes at each stage.

In [ ]:
U, Up, Um = pipeline(even0, end1)
print("union      : %2d states" % len(U["Q"]))
print("pruned     : %2d states" % len(Up["Q"]))
print("minimized  : %2d states" % len(Um["Q"]))
assert len(Um["Q"]) <= len(Up["Q"]) <= len(U["Q"])

All three accept the same language.

In [ ]:
from itertools import product
strs = [''.join(p) for k in range(11) for p in product('01', repeat=k)]
spec = lambda s: (s.count('0') % 2 == 0) or s.endswith('1')
for name, X in [('union', U), ('pruned', Up), ('minimized', Um)]:
    bad = [s for s in strs if accepts_dfa(X, s) != spec(s)]
    print("%-10s mismatches : %d" % (name, len(bad)))
    assert not bad

**The punchline:** equivalent but not isomorphic.

In [ ]:
print("langeq_dfa(U, Um) :", langeq_dfa(U, Um))
print("iso_dfa(U, Um)    :", iso_dfa(U, Um))
assert langeq_dfa(U, Um)
assert not iso_dfa(U, Um)
print("\nSame language (%d states vs %d) -- different machines."
      % (len(U["Q"]), len(Um["Q"])))

Minimizing both sides restores isomorphism, as Myhill&ndash;Nerode promises.

In [ ]:
print("iso_dfa(min(U), min(Um)) :", iso_dfa(min_dfa(U), min_dfa(Um)))
assert iso_dfa(min_dfa(U), min_dfa(Um))

## 4. Animation

The minimized union &mdash; the whole pipeline's output.

*(The `display(HTML(...))` line loads the toolbar's font-awesome icons. Keep it last in the cell &mdash; it must be there for the controls to appear.)*

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(min_dfa(pruneUnreach(union_dfa(even0, end1))), FuseEdges=True)
display(HTML('<link rel="stylesheet" href="//stackpath.bootstrapcdn.com/font-awesome/4.7.0/css/font-awesome.min.css"/>'))

## 5. Exercises


1. Run the same pipeline with `intersect_dfa`. How do the sizes compare?
2. Does pruning before minimizing change the final answer? Does it change the cost?
3. Find two DFA whose union does **not** shrink at all under minimization.

In [ ]:
# Your work for the exercises above.